In [ ]:
import os
import sys

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

PROJECT_ROOT = "/Users/saratramontana/Documents/GitHub/test_segmentation_model"
os.chdir(PROJECT_ROOT)

sys.path.append(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "external", "ACSNet"))
sys.path.append(os.path.join(PROJECT_ROOT, "external", "OvaMTA"))

import torch
import wandb
import lightning as L
import pandas as pd
import matplotlib.pyplot as plt

from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint

from new_data.data_loader import train_df, val_df
from training.factories import build_dataloaders, build_lightning_module

In [ ]:
CONFIG = {
    "models_to_run": [
        "baseline",
        "acsnet",
        # "ovamta_seg",
        # "ovamta_diag",
    ],

    "lrs": [1e-2, 1e-3, 1e-4],

    "batch_size": 8,
    "max_epochs": 10,

    "wandb_project": "model_comparison",

    "acsnet_seg_loss_mode": "ce_lossnet",
    "acsnet_gamma": 0.2,
}

In [ ]:
def train_one_model(model_name, lr):
    train_loader, val_loader = build_dataloaders(
        model_name=model_name,
        train_df=train_df,
        val_df=val_df,
        batch_size=CONFIG["batch_size"],
    )

    model = build_lightning_module(
        model_name=model_name,
        lr=lr,
        train_df=train_df,
        seg_loss_mode=CONFIG["acsnet_seg_loss_mode"],
        gamma=CONFIG["acsnet_gamma"],
    )

    wandb_logger = WandbLogger(
        project=CONFIG["wandb_project"],
        name=f"{model_name}_lr_{lr}",
        log_model=False,
    )

    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    trainer = L.Trainer(
        max_epochs=CONFIG["max_epochs"],
        log_every_n_steps=1,
        logger=wandb_logger,
        callbacks=[checkpoint_callback],
    )

    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader,
    )

    metrics = trainer.callback_metrics

    result = {
        "model": model_name,
        "lr": lr,
        "best_ckpt_path": checkpoint_callback.best_model_path,
    }

    metric_names = [
        "val_loss",
        "val_seg_loss",
        "val_cls_loss",
        "val_cls_acc",
        "val_seg_dice",
        "val_seg_iou",
        "val_cls_f1",
        "val_cls_precision",
        "val_cls_recall",
        "val_cls_auc",
    ]

    for name in metric_names:
        value = metrics.get(name)

        if value is None:
            result[name] = None
        else:
            result[name] = float(value.detach().cpu())

    wandb.finish()

    return result

In [ ]:
all_results = []

for model_name in CONFIG["models_to_run"]:
    for lr in CONFIG["lrs"]:
        print(f"\nRunning model={model_name}, lr={lr}")

        result = train_one_model(
            model_name=model_name,
            lr=lr,
        )

        all_results.append(result)

comparison_df = pd.DataFrame(all_results)
comparison_df

In [ ]:
best_by_model = (
    comparison_df
    .sort_values("val_loss", ascending=True)
    .groupby("model", as_index=False)
    .first()
)

best_by_model

In [ ]:
metrics_to_plot = [
    "val_seg_dice",
    "val_seg_iou",
    "val_cls_f1",
    "val_cls_auc",
]

best_by_model.plot(
    x="model",
    y=metrics_to_plot,
    kind="bar",
    figsize=(10, 5),
)

plt.title("Best model comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()